# Wildfire Data Collection and Cleaning

## Objectives

* Load raw EFFIS wildfire data (burnt area + number of fires)
* Inspect for missing values and structural gaps
* Reshape from wide to long format
* Save cleaned dataset for EDA

## Inputs

* inputs/raw/burnt_area_ha_1980_2024.csv
* inputs/raw/number_of_fires_1980_2024.csv

## Outputs

* inputs/processed/wildfires_long_format.csv

## Additional Comments

* Source: European Forest Fire Information System (EFFIS), Copernicus Emergency Management Service (European Commission Joint Research Centre)
* Data download page: https://forest-fire.emergency.copernicus.eu/applications/data-and-services
* Countries joined EFFIS reporting at different times, so missing values before a country's join year are expected and not treated as data quality issues

---

In [1]:
import pandas as pd

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [2]:
import os
current_dir = os.getcwd()
current_dir

'/Users/tildeholmqvist/Documents/VS_Code_Tilde/DA_project_3/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chdir() defines the new current directory

In [3]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [4]:
current_dir = os.getcwd()
current_dir

'/Users/tildeholmqvist/Documents/VS_Code_Tilde/DA_project_3'

# Section 1: Load Raw Data

Load the two raw EFFIS CSV files (burnt area and number of fires) and do an initial
shape/structure check before any cleaning is applied.

In [5]:
burnt_area = pd.read_csv("inputs/raw/burnt_area_ha_1980_2024.csv")
num_fires = pd.read_csv("inputs/raw/number_of_fires_1980_2024.csv")

print("Burnt area shape:", burnt_area.shape)
print("Number of fires shape:", num_fires.shape)
burnt_area.head()

Burnt area shape: (45, 32)
Number of fires shape: (45, 32)


,Year,PRT,ESP,FRA,ITA,GRC,DZA,AUT,BGR,HRV,...,NOR,POL,ROU,SRB,SVK,SVN,SWE,CHE,TUR,UKR
0,1980,44251,263017.0,22176,143919,32965.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1981,89798,298288.0,27711,229850,81417.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1982,39556,152903.0,55145,130456,27372.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1983,47811,108100.0,53729,212678,19613.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1984,52710,165119.0,27202,75272,33655.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---

# Section 2: Inspect Data Quality

Check for missing values and identify which years/countries have structural gaps
in EFFIS reporting.

In [6]:
print("Years covered:", burnt_area['Year'].min(), "-", burnt_area['Year'].max())
print()
print("Missing values per country (burnt area):")
print(burnt_area.drop(columns='Year').isna().sum().sort_values())

Years covered: 1980 - 2024

Missing values per country (burnt area):
PRT     0
ESP     0
FRA     0
ITA     0
GRC     0
CHE    10
ROU    10
POL    10
MAR    10
TUR    10
LVA    10
DEU    11
BGR    11
LTU    12
HRV    12
AUT    13
CZE    15
FIN    16
SWE    18
SVK    19
EST    20
HUN    20
CYP    20
NOR    21
SVN    22
MKD    27
UKR    27
SRB    29
DZA    33
LBN    36
NLD    37
dtype: int64


Spain, Portugal, France, Italy, and Greece have complete data from 1980 onwards,
confirming they are well-suited as the focus countries for this analysis. Countries
with higher missing counts joined EFFIS reporting later and will be excluded from
any year before their reporting began, rather than having values imputed.

---

# Section 3: Reshape data to Long Format

Convert both datasets from wide format (one column per country) to long format
(one row per year-country combination), then merge them into a single tidy dataset.

In [7]:
burnt_area_long = burnt_area.melt(id_vars='Year', var_name='country_iso3', value_name='burnt_area_ha')

burnt_area_long.head()

,Year,country_iso3,burnt_area_ha
0,1980,PRT,44251.0
1,1981,PRT,89798.0
2,1982,PRT,39556.0
3,1983,PRT,47811.0
4,1984,PRT,52710.0


In [8]:
print("Total rows:", burnt_area_long.shape[0])
print("Number of unique countries:", burnt_area_long['country_iso3'].nunique())
print("All countries:", sorted(burnt_area_long['country_iso3'].unique()))

Total rows: 1395
Number of unique countries: 31
All countries: ['AUT', 'BGR', 'CHE', 'CYP', 'CZE', 'DEU', 'DZA', 'ESP', 'EST', 'FIN', 'FRA', 'GRC', 'HRV', 'HUN', 'ITA', 'LBN', 'LTU', 'LVA', 'MAR', 'MKD', 'NLD', 'NOR', 'POL', 'PRT', 'ROU', 'SRB', 'SVK', 'SVN', 'SWE', 'TUR', 'UKR']


The reshaped dataset confirms all 31 countries are present, with 45 rows each
(1980-2024) — the `.head()` preview only showed Portugal because the melted rows
follow the original column order, not because other countries were dropped.

In [9]:
num_fires_long = num_fires.melt(id_vars='Year', var_name='country_iso3', value_name='number_of_fires')

print(num_fires_long.shape)
num_fires_long.head()

(1395, 3)


,Year,country_iso3,number_of_fires
0,1980,PRT,2349.0
1,1981,PRT,6730.0
2,1982,PRT,3626.0
3,1983,PRT,4539.0
4,1984,PRT,7356.0


In [10]:
print("Total rows:", num_fires_long.shape[0])
print("Number of unique countries:", num_fires_long['country_iso3'].nunique())
print("All countries:", sorted(num_fires_long['country_iso3'].unique()))

Total rows: 1395
Number of unique countries: 31
All countries: ['AUT', 'BGR', 'CHE', 'CYP', 'CZE', 'DEU', 'DZA', 'ESP', 'EST', 'FIN', 'FRA', 'GRC', 'HRV', 'HUN', 'ITA', 'LBN', 'LTU', 'LVA', 'MAR', 'MKD', 'NLD', 'NOR', 'POL', 'PRT', 'ROU', 'SRB', 'SVK', 'SVN', 'SWE', 'TUR', 'UKR']


Same structure and country count as the burnt area dataset, confirming both are
ready to be merged on `Year` and `country_iso3`.

## Merge burnt area and number of fires into a single dataset

Both long-format tables share the same structure (Year, country_iso3), so they can
be merged into one tidy dataset with both indicators as separate columns.

In [11]:
wildfires_long = pd.merge(burnt_area_long, num_fires_long, on=['Year', 'country_iso3'], how='outer')

print(wildfires_long.shape)
wildfires_long.head()

(1395, 4)


,Year,country_iso3,burnt_area_ha,number_of_fires
0,1980,PRT,44251.0,2349.0
1,1981,PRT,89798.0,6730.0
2,1982,PRT,39556.0,3626.0
3,1983,PRT,47811.0,4539.0
4,1984,PRT,52710.0,7356.0


## Add readable country names

ISO3 codes are precise but not reader-friendly for a non-technical dashboard
audience, so they are mapped to full country names.

In [12]:
iso3_to_name = {
    'PRT': 'Portugal', 'ESP': 'Spain', 'FRA': 'France', 'ITA': 'Italy', 'GRC': 'Greece',
    'DZA': 'Algeria', 'AUT': 'Austria', 'BGR': 'Bulgaria', 'HRV': 'Croatia', 'CYP': 'Cyprus',
    'CZE': 'Czechia', 'EST': 'Estonia', 'FIN': 'Finland', 'DEU': 'Germany', 'HUN': 'Hungary',
    'LVA': 'Latvia', 'LBN': 'Lebanon', 'LTU': 'Lithuania', 'MAR': 'Morocco', 'NLD': 'Netherlands',
    'MKD': 'North Macedonia', 'NOR': 'Norway', 'POL': 'Poland', 'ROU': 'Romania', 'SRB': 'Serbia',
    'SVK': 'Slovakia', 'SVN': 'Slovenia', 'SWE': 'Sweden', 'CHE': 'Switzerland', 'TUR': 'Turkey',
    'UKR': 'Ukraine'
}

wildfires_long['country_name'] = wildfires_long['country_iso3'].map(iso3_to_name)

unmapped = wildfires_long[wildfires_long['country_name'].isna()]['country_iso3'].unique()
print("Unmapped ISO3 codes (should be empty):", unmapped)

Unmapped ISO3 codes (should be empty): []


## Drop rows with no data at all

Some (year, country) combinations have no data for either indicator, since that
country had not yet joined EFFIS reporting. These rows carry no information and
are removed. Rows with only partial data are kept, since they are still informative.

In [13]:
before = len(wildfires_long)
wildfires_long = wildfires_long.dropna(subset=['burnt_area_ha', 'number_of_fires'], how='all')
after = len(wildfires_long)

print(f"Dropped {before - after} fully-empty rows. Remaining: {after} rows.")

Dropped 473 fully-empty rows. Remaining: 922 rows.


---

# Section 4: Save Processed Dataset

In [14]:
wildfires_long.to_csv("inputs/processed/wildfires_long_format.csv", index=False)
print(f"Saved {len(wildfires_long)} rows to inputs/processed/wildfires_long_format.csv")

Saved 922 rows to inputs/processed/wildfires_long_format.csv


# Conclusions and Next Steps

This notebook loaded, inspected, reshaped, and cleaned the raw EFFIS wildfire data
for 31 countries (1980-2024), producing a tidy long-format dataset with 922 rows.

**Key takeaways:**
* Spain, Portugal, France, Italy, and Greece have complete reporting from 1980
  onwards, confirming they are well-suited as the focus countries for this project
* Countries with gaps in early years had not yet joined EFFIS reporting; these
  rows were dropped rather than imputed, to avoid fabricating data

**Output:** `inputs/processed/wildfires_long_format.csv`, ready for exploratory
data analysis.

**Next step:** `02_eda.ipynb` — exploratory analysis of wildfire trends, with a
focus on Spain, Portugal, France, and Greece in the context of the 2025-2026
wildfire crisis.